# New Mexico 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready county-level table for New Mexico, 2008 by merging the presidential primary and presidential general election results, and then derive summary stats (party totals). Note that, in the primary dataset, the presidential election data only recorded values for the Republican, missing the Democratic. Thus, this dataset might not be good for complacency modeling.

**Output**: A single CSV where each row is a county and columns include:

- Primary per-candidate vote counts (prefixed with `pri_`)
- General per-candidate vote counts (prefixed with `gen_`)
- Party totals: `rep_primary_total`, `rep_general_total`, `dem_general_total`, `lib_general_total`, `grn_general_total`, `cst_general_total`, `ind_general_total`

**Last Updated**: 2025/10/22

## 0. Library Import

In [1]:
import re 
import pandas as pd
import numpy as numpy
from pathlib import Path

/Users/amourtu1934/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Inputs & Parameters

In [53]:
# NM 2008 dataset path
PRIMARY_PATH = r"../../data/raw/2008/NM/20080603__nm__primary.csv"
GENERAL_PATH = r"../../data/raw/2008/NM/20081104__nm__general.csv"

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/NM/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

### a. Primary Election Dataset

In [54]:
# Load primary data
primary_df = pd.read_csv(PRIMARY_PATH)
primary_df.head(DISPLAY_ROWS)

,county,office,district,party,candidate,votes
0,Bernalillo,UNITED STATES SENATOR,NaN,DEMOCRAT,TOM UDALL,45321
1,Catron,UNITED STATES SENATOR,NaN,DEMOCRAT,TOM UDALL,229
2,Chaves,UNITED STATES SENATOR,NaN,DEMOCRAT,TOM UDALL,1968
3,Cibola,UNITED STATES SENATOR,NaN,DEMOCRAT,TOM UDALL,3165
4,Colfax,UNITED STATES SENATOR,NaN,DEMOCRAT,TOM UDALL,1597
5,Curry,UNITED STATES SENATOR,NaN,DEMOCRAT,TOM UDALL,1392
6,De Baca,UNITED STATES SENATOR,NaN,DEMOCRAT,TOM UDALL,340
7,Dona Ana,UNITED STATES SENATOR,NaN,DEMOCRAT,TOM UDALL,8332
8,Eddy,UNITED STATES SENATOR,NaN,DEMOCRAT,TOM UDALL,3577
9,Grant,UNITED STATES SENATOR,NaN,DEMOCRAT,TOM UDALL,3740


In [55]:
# Different values in 'office' column
primary_df["office"].value_counts()

office
STATE REPRESENTATIVE              344
UNITED STATES REPRESENTATIVE      305
STATE SENATOR                     258
UNITED STATES SENATOR             102
PRESIDENT OF THE UNITED STATES     68
JUSTICE OF THE SUPREME COURT       34
Name: count, dtype: int64

In [56]:
# Only keep rows where 'office' is 'PRESIDENT OF THE UNITED STATES'
primary_df = primary_df[primary_df["office"] == "PRESIDENT OF THE UNITED STATES"]
primary_df.head(DISPLAY_ROWS)

,county,office,district,party,candidate,votes
603,Bernalillo,PRESIDENT OF THE UNITED STATES,NaN,REPUBLICAN,JOHN MCCAIN,35706
604,Catron,PRESIDENT OF THE UNITED STATES,NaN,REPUBLICAN,JOHN MCCAIN,399
605,Chaves,PRESIDENT OF THE UNITED STATES,NaN,REPUBLICAN,JOHN MCCAIN,5251
606,Cibola,PRESIDENT OF THE UNITED STATES,NaN,REPUBLICAN,JOHN MCCAIN,736
607,Colfax,PRESIDENT OF THE UNITED STATES,NaN,REPUBLICAN,JOHN MCCAIN,688
608,Curry,PRESIDENT OF THE UNITED STATES,NaN,REPUBLICAN,JOHN MCCAIN,2456
609,De Baca,PRESIDENT OF THE UNITED STATES,NaN,REPUBLICAN,JOHN MCCAIN,187
610,Dona Ana,PRESIDENT OF THE UNITED STATES,NaN,REPUBLICAN,JOHN MCCAIN,5710
611,Eddy,PRESIDENT OF THE UNITED STATES,NaN,REPUBLICAN,JOHN MCCAIN,2796
612,Grant,PRESIDENT OF THE UNITED STATES,NaN,REPUBLICAN,JOHN MCCAIN,1224


In [57]:
# Primary data shape when only considering President/VicePresident
primary_df.shape

(68, 6)

In [58]:
# Number of missing values in each column
primary_df.isna().sum()

county        2
office        0
district     68
party         0
candidate     0
votes         0
dtype: int64

In [59]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the district column since it's all missing values
primary_df = primary_df.drop(columns=["office", "district"]).reset_index(drop=True)
primary_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,Bernalillo,REPUBLICAN,JOHN MCCAIN,35706
1,Catron,REPUBLICAN,JOHN MCCAIN,399
2,Chaves,REPUBLICAN,JOHN MCCAIN,5251
3,Cibola,REPUBLICAN,JOHN MCCAIN,736
4,Colfax,REPUBLICAN,JOHN MCCAIN,688
5,Curry,REPUBLICAN,JOHN MCCAIN,2456
6,De Baca,REPUBLICAN,JOHN MCCAIN,187
7,Dona Ana,REPUBLICAN,JOHN MCCAIN,5710
8,Eddy,REPUBLICAN,JOHN MCCAIN,2796
9,Grant,REPUBLICAN,JOHN MCCAIN,1224


Note that there are two missing values in `county`. We thus want to look at these specific two in order to come up with a plan for these missing fields.

In [60]:
# Observation with missing value in `county`
primary_df.loc[primary_df["county"].isna()]

,county,party,candidate,votes
33,NaN,REPUBLICAN,JOHN MCCAIN,95378
67,NaN,REPUBLICAN,RON PAUL,15561


In [61]:
# Different candidates in primary_df
primary_df["candidate"].value_counts()

candidate
JOHN MCCAIN    34
RON PAUL       34
Name: count, dtype: int64

Given there are only two candidates, and there are 33 counties in New Mexico, our guess is that the two rows with missing `county` value are total vote counts for each candidate. We thus check this claim as follows:

In [62]:
# Total votes for each candidate across all counties except row with missing value in county
primary_df.loc[
    primary_df['county'].notna()        # Filter out rows with missing value in `county`
].groupby('candidate', sort=True)['votes'].sum()

candidate
JOHN MCCAIN    95378
RON PAUL       15561
Name: votes, dtype: int64

The numbers agree, thus confirm our suspect on the purpose of such rows. Thus, we can drop the two observations with missing values in `county` as we ultimately pivot the table and recount it anyway.

In [63]:
# Drop observation with misisng value in `county`
primary_df = primary_df[primary_df["county"].notna()].reset_index(drop=True)

# Shape of primary_df after dropping such observation
primary_df.shape

(66, 4)

In [64]:
# Quick peek of the current primary_df
primary_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,Bernalillo,REPUBLICAN,JOHN MCCAIN,35706
1,Catron,REPUBLICAN,JOHN MCCAIN,399
2,Chaves,REPUBLICAN,JOHN MCCAIN,5251
3,Cibola,REPUBLICAN,JOHN MCCAIN,736
4,Colfax,REPUBLICAN,JOHN MCCAIN,688
5,Curry,REPUBLICAN,JOHN MCCAIN,2456
6,De Baca,REPUBLICAN,JOHN MCCAIN,187
7,Dona Ana,REPUBLICAN,JOHN MCCAIN,5710
8,Eddy,REPUBLICAN,JOHN MCCAIN,2796
9,Grant,REPUBLICAN,JOHN MCCAIN,1224


In [65]:
# List out all the parties in the general election data
primary_df["party"].value_counts()

party
REPUBLICAN    66
Name: count, dtype: int64

This is interesting as we only have data for the Republican party in the primary election in New Mexico in 2008, where the state actually had both parties' presidential contest. Thus, we cannot use this to do complacency analysis as we intended to.

In [66]:
# Data type of each column in primary_df
primary_df.dtypes

county       object
party        object
candidate    object
votes         int64
dtype: object

In [67]:
# Final look at the cleaned primary_df
primary_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,Bernalillo,REPUBLICAN,JOHN MCCAIN,35706
1,Catron,REPUBLICAN,JOHN MCCAIN,399
2,Chaves,REPUBLICAN,JOHN MCCAIN,5251
3,Cibola,REPUBLICAN,JOHN MCCAIN,736
4,Colfax,REPUBLICAN,JOHN MCCAIN,688
5,Curry,REPUBLICAN,JOHN MCCAIN,2456
6,De Baca,REPUBLICAN,JOHN MCCAIN,187
7,Dona Ana,REPUBLICAN,JOHN MCCAIN,5710
8,Eddy,REPUBLICAN,JOHN MCCAIN,2796
9,Grant,REPUBLICAN,JOHN MCCAIN,1224


In [68]:
# Shape after preprocessing
primary_df.shape

(66, 4)

### b. General Election Dataset

In [23]:
# Load general data
general_df = pd.read_csv(GENERAL_PATH)
general_df.head(DISPLAY_ROWS)

,county,office,district,party,candidate,votes
0,Bernalillo,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,171556
1,Catron,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,664
2,Chaves,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,8197
3,Cibola,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,5827
4,Colfax,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,3490
5,Curry,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,4670
6,De Baca,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,359
7,Dona Ana,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,40282
8,Eddy,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,7351
9,Grant,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,8142


In [24]:
# Different values in 'office' column
general_df["office"].value_counts()

office
STATE REPRESENTATIVE                                 277
PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES    204
STATE SENATOR                                        179
UNITED STATES REPRESENTATIVE                          84
UNITED STATES SENATOR                                 68
JUSTICE OF THE SUPREME COURT                          34
Name: count, dtype: int64

In [25]:
# Only keep rows where 'office' includes "PRESIDENT"
general_df = general_df[general_df["office"] == "PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES"]
general_df.head(DISPLAY_ROWS)

,county,office,district,party,candidate,votes
0,Bernalillo,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,171556
1,Catron,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,664
2,Chaves,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,8197
3,Cibola,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,5827
4,Colfax,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,3490
5,Curry,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,4670
6,De Baca,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,359
7,Dona Ana,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,40282
8,Eddy,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,7351
9,Grant,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES,NaN,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,8142


In [26]:
# General data shape when only considering President/VicePresident
general_df.shape

(204, 6)

In [27]:
# Number of missing values in each column
general_df.isna().sum()

county         6
office         0
district     204
party          0
candidate      0
votes          0
dtype: int64

In [29]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the "district" column since it's all missing values
general_df = general_df.drop(columns=["office", "district"]).reset_index(drop=True)
general_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,Bernalillo,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,171556
1,Catron,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,664
2,Chaves,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,8197
3,Cibola,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,5827
4,Colfax,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,3490
5,Curry,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,4670
6,De Baca,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,359
7,Dona Ana,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,40282
8,Eddy,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,7351
9,Grant,DEMOCRATIC PARTY,BARACK OBAMA and JOE BIDEN,8142


In [30]:
# Candidates in general_df
general_df["candidate"].value_counts()

candidate
BARACK OBAMA and JOE BIDEN            34
JOHN MCCAIN and SARAH PALIN           34
CYNTHIA MCKINNEY and ROSA CLEMENTE    34
CHUCK BALDWIN and DARRELL CASTLE      34
RALPH NADER and MATT GONZALEZ         34
BOB BARR and WAYNE A. ROOT            34
Name: count, dtype: int64

Now, each row’s candidate value now contains two names: presidential first, vice-presidential second, which is separated by an "and" We’ll split on the "and" and retain only the presidential name.

In [31]:
# Keep only the presidential candidate in the "candidate" column
general_df["candidate"] = (
    general_df["candidate"]
      .str.split(r"(?i)\s*(?:andf|and|&|/|/)\s*", n=1, expand=True)[0]
      .str.strip()
)

# Candidates in general_df
general_df["candidate"].value_counts()

candidate
BARACK OBAMA        34
JOHN MCCAIN         34
CYNTHIA MCKINNEY    34
CHUCK BALDWIN       34
RALPH NADER         34
BOB BARR            34
Name: count, dtype: int64

There are 6 different candidates and there are 6 missing values in `county` column. We suspect the same reason as above: rows with missing county is the total row. We can check that as follows:

In [35]:
# Observation with missing value in `county`
general_df.loc[general_df["county"].isna()]

,county,party,candidate,votes
33,NaN,DEMOCRATIC PARTY,BARACK OBAMA,472422
67,NaN,REPUBLICAN PARTY,JOHN MCCAIN,346832
101,NaN,GREEN PARTY,CYNTHIA MCKINNEY,1552
135,NaN,CONSTITUTION PARTY,CHUCK BALDWIN,1597
169,NaN,INDEPENDENT PARTY,RALPH NADER,5327
203,NaN,LIBERTARIAN PARTY,BOB BARR,2428


In [36]:
# Total votes for each candidate across all counties except row with missing value in county
general_df.loc[
    general_df['county'].notna()        # Filter out rows with missing value in `county`
].groupby('candidate')['votes'].sum()

candidate
BARACK OBAMA        472422
BOB BARR              2428
CHUCK BALDWIN         1597
CYNTHIA MCKINNEY      1552
JOHN MCCAIN         346832
RALPH NADER           5327
Name: votes, dtype: int64

The numbers again match. Thus, we can also drop these rows.

In [38]:
# Drop observation with misisng value in `county`
general_df = general_df[general_df["county"].notna()].reset_index(drop=True)

# Shape of primary_df after dropping such observation
general_df.shape

(198, 4)

In [39]:
# List out all the parties in the general election data
general_df["party"].value_counts()

party
DEMOCRATIC PARTY      33
REPUBLICAN PARTY      33
GREEN PARTY           33
CONSTITUTION PARTY    33
INDEPENDENT PARTY     33
LIBERTARIAN PARTY     33
Name: count, dtype: int64

Now, we indeed have different parties, which include the two main parties that we are interested in, Democratic and Republican. We can thus pivot this dataframe and add the total column as normal.

In [41]:
# Data type of each column in general_df
general_df.dtypes

county       object
party        object
candidate    object
votes         int64
dtype: object

In [40]:
# Final look at the cleaned general_df
general_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,Bernalillo,DEMOCRATIC PARTY,BARACK OBAMA,171556
1,Catron,DEMOCRATIC PARTY,BARACK OBAMA,664
2,Chaves,DEMOCRATIC PARTY,BARACK OBAMA,8197
3,Cibola,DEMOCRATIC PARTY,BARACK OBAMA,5827
4,Colfax,DEMOCRATIC PARTY,BARACK OBAMA,3490
5,Curry,DEMOCRATIC PARTY,BARACK OBAMA,4670
6,De Baca,DEMOCRATIC PARTY,BARACK OBAMA,359
7,Dona Ana,DEMOCRATIC PARTY,BARACK OBAMA,40282
8,Eddy,DEMOCRATIC PARTY,BARACK OBAMA,7351
9,Grant,DEMOCRATIC PARTY,BARACK OBAMA,8142


In [42]:
# Shape after preprocessing
general_df.shape

(198, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: maps common forms (e.g., “Democratic”, “Republican”) to keys dem/rep so column names are stable
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [74]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Normalize party names: Democratic -> dem, Republican -> rep
    """
    return(s.str.strip()
           .str.capitalize()
           .map({
                "Democratic"          : "dem",
                "Democratic party"    : "dem", 
                "Republican"          : "rep",
                "Republican party"    : "rep",
                "Green party"         : "grn",
                "Constitution party"  : "cst",
                "Libertarian party"   : "lib",
                "Independent party"   : "ind"
               })
           .fillna(s.str.strip().str.lower()))      # For defensive purposes only, would not expect other parties

In [75]:
SUFFIXES = {
    "JR","SR","JNR","SNR",
    "II","III","IV","V","VI","VII","VIII","IX","X","XI","XII"
}

def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values
    
    # Remove suffixes
    raw = str(name).strip()

    # If a comma exists, treat as 'LAST, FIRST ...'
    if "," in raw:
        last_part = raw.split(",", 1)[0]
        last_part = re.sub(r"[^A-Za-z0-9\s]+", "", last_part).strip().upper()
        tokens = last_part.split()
        return tokens[-1] if tokens else "UNKNOWN"

    # Otherwise: remove punctuation, split, then drop trailing suffixes
    tokens = re.sub(r"[^A-Za-z0-9\s]+", "", raw).strip().upper().split()
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()
    return tokens[-1] if tokens else "UNKNOWN"

In [76]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [77]:
# Primary dataframe pivot
primary_pivot = pivot_wide(primary_df, prefix="pri")
primary_pivot.head(DISPLAY_ROWS)

,county,pri_rep_MCCAIN,pri_rep_PAUL
0,Bernalillo,35706,6417
1,Catron,399,143
2,Chaves,5251,613
3,Cibola,736,128
4,Colfax,688,108
5,Curry,2456,211
6,De Baca,187,30
7,Dona Ana,5710,811
8,Eddy,2796,226
9,Grant,1224,223


In [78]:
# Primary dataframe shape after pivot
primary_pivot.shape

(33, 3)

In [79]:
# General dataframe pivot
general_pivot = pivot_wide(general_df, prefix="gen")
general_pivot.head(DISPLAY_ROWS)

,county,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_ind_NADER,gen_lib_BARR,gen_rep_MCCAIN
0,Bernalillo,501,171556,481,1779,940,110521
1,Catron,20,664,6,14,10,1398
2,Chaves,35,8197,34,141,54,13651
3,Cibola,31,5827,23,73,12,3131
4,Colfax,10,3490,12,58,9,2805
5,Curry,23,4670,21,92,33,9599
6,De Baca,0,359,0,7,2,676
7,Dona Ana,105,40282,138,466,221,28068
8,Eddy,41,7351,28,114,59,12500
9,Grant,41,8142,34,92,40,5406


In [80]:
# General dataframe shape after pivot
general_pivot.shape

(33, 7)

## 4. Merge Dataframes

Before merging, we verify that county names match across primary and general:

In [81]:
# Check if county names match between primary_df and general_df
primary_counties = set(primary_pivot["county"].unique())
general_counties = set(general_pivot["county"].unique())
common_counties = primary_counties.intersection(general_counties)
print(f"Number of common counties: {len(common_counties)} out of {len(primary_counties)}")

Number of common counties: 33 out of 33


Great. Since we know that all counties name are matched, we don't need to perform further data preprocessing to match the county names. Thus, we can now merge them:

In [82]:
# Merge primary and general dataframes on 'county'
merged_df = primary_pivot.merge(general_pivot, on="county", how="inner").fillna(0)    # There should be no missing values to fill with 0
merged_df.head(DISPLAY_ROWS)

,county,pri_rep_MCCAIN,pri_rep_PAUL,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_ind_NADER,gen_lib_BARR,gen_rep_MCCAIN
0,Bernalillo,35706,6417,501,171556,481,1779,940,110521
1,Catron,399,143,20,664,6,14,10,1398
2,Chaves,5251,613,35,8197,34,141,54,13651
3,Cibola,736,128,31,5827,23,73,12,3131
4,Colfax,688,108,10,3490,12,58,9,2805
5,Curry,2456,211,23,4670,21,92,33,9599
6,De Baca,187,30,0,359,0,7,2,676
7,Dona Ana,5710,811,105,40282,138,466,221,28068
8,Eddy,2796,226,41,7351,28,114,59,12500
9,Grant,1224,223,41,8142,34,92,40,5406


In [83]:
# Statistics check on merged dataframe 
merged_df.describe()

,pri_rep_MCCAIN,pri_rep_PAUL,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_ind_NADER,gen_lib_BARR,gen_rep_MCCAIN
count,33.000000,33.000000,33.000000,33.000000,33.000000,33.000000,33.000000,33.000000
mean,2890.242424,471.545455,48.393939,14315.818182,47.030303,161.424242,73.575758,10510.060606
std,6165.005064,1107.439786,87.926084,30728.112282,85.859436,312.862892,164.781679,19546.027876
min,143.000000,14.000000,0.000000,260.000000,0.000000,5.000000,0.000000,358.000000
25%,525.000000,97.000000,11.000000,2303.000000,13.000000,48.000000,12.000000,2478.000000
50%,1127.000000,206.000000,25.000000,5108.000000,23.000000,74.000000,34.000000,4086.000000
75%,3087.000000,380.000000,41.000000,12703.000000,40.000000,114.000000,59.000000,12806.000000
max,35706.000000,6417.000000,501.000000,171556.000000,481.000000,1779.000000,940.000000,110521.000000



Now, we will add party totals columns for general totals:

- Primary totals:
    * `rep_primary_total` = sum of all `pri_rep_*` columns
    
- General totals:
    * `dem_general_total` = sum of all `gen_dem_*` columns
    * `rep_general_total` = sum of all `gen_rep_*` columns
    * `grn_general_total` = sum of all `gen_grn_*` columns
    * `cst_general_total` = sum of all `gen_cst_*` columns
    * `lib_general_total` = sum of all `gen_lib_*` columns
    * `ind_general_total` = sum of all `gen_ind_*` columns`

In [84]:
# Add party totals for general election
dem_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_dem")]
rep_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_rep")] 
grn_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_gre")]
cst_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_cst")]
lib_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_lib")]
ind_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_ind")]

general_pivot["dem_general_total"] = general_pivot[dem_general_cols].sum(axis=1) if dem_general_cols else 0
general_pivot["rep_general_total"] = general_pivot[rep_general_cols].sum(axis=1) if rep_general_cols else 0
general_pivot["grn_general_total"] = general_pivot[grn_general_cols].sum(axis=1) if grn_general_cols else 0
general_pivot["cst_general_total"] = general_pivot[cst_general_cols].sum(axis=1) if cst_general_cols else 0
general_pivot["lib_general_total"] = general_pivot[lib_general_cols].sum(axis=1) if lib_general_cols else 0
general_pivot["ind_general_total"] = general_pivot[ind_general_cols].sum(axis=1) if ind_general_cols else 0

In [87]:
# Print out all the column names in the final dataframe
print("Final columns in the cleaned merged dataframe:")
merged_df.columns

Final columns in the cleaned merged dataframe:


Index(['county', 'pri_rep_MCCAIN', 'pri_rep_PAUL', 'gen_cst_BALDWIN',
       'gen_dem_OBAMA', 'gen_grn_MCKINNEY', 'gen_ind_NADER', 'gen_lib_BARR',
       'gen_rep_MCCAIN'],
      dtype='object')

In [88]:
# Preview the merged_df dataframe with totals
merged_df.head(DISPLAY_ROWS)

,county,pri_rep_MCCAIN,pri_rep_PAUL,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_ind_NADER,gen_lib_BARR,gen_rep_MCCAIN
0,Bernalillo,35706,6417,501,171556,481,1779,940,110521
1,Catron,399,143,20,664,6,14,10,1398
2,Chaves,5251,613,35,8197,34,141,54,13651
3,Cibola,736,128,31,5827,23,73,12,3131
4,Colfax,688,108,10,3490,12,58,9,2805
5,Curry,2456,211,23,4670,21,92,33,9599
6,De Baca,187,30,0,359,0,7,2,676
7,Dona Ana,5710,811,105,40282,138,466,221,28068
8,Eddy,2796,226,41,7351,28,114,59,12500
9,Grant,1224,223,41,8142,34,92,40,5406


Now, we save the cleaned dataframe into the processed directory.

In [89]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(OUTPUT_PATH + "NM.csv", index=False)